In [1]:
import sys

assert sys.version_info >= (3,10)

In [2]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
import matplotlib.pyplot as  plt

plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('font', size=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "07_Euler_Bernoulli_Beam_Equation_with_Higher_order"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [5]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Exact solution:
Got this exact solution by integrating the bernoulli equation and substituting the boundary conditions for cantilever beam.

In [7]:
def exact_solution(x):
    return -(1/24) * x ** 4 + (1/6) * x ** 3 - (1/4) * x ** 2

Creating the geometry for the beam of unit length

In [8]:
geom =  dde.geometry.Interval(0, 1)


Creating a Partial differential equation for external distributed load with 1KN/m

In [9]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    dy_xxxx = dde.grad.hessian(dy_xx, x, i=0, j=0)
    return dy_xxxx + 1

creating the boundary conditions at x = 0

In [11]:
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], 0)

bc_u_0 = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)
bc_du_0 = dde.icbc.NeumannBC(geom, lambda x:0, boundary_left)

Creating the Boundary Conditions at x = 1

In [12]:
def boundary_right(x, on_boundary):
    return on_boundary and np.isclose(x[0], 1)

def boundary_second_derivative(x, y, X):
    return dde.grad.hessian(y, x, i=0, j=0)

bc_obc1 = dde.icbc.OperatorBC(
    geom, boundary_second_derivative,
    boundary_right
)

def boundary_third_derivative(x, y, X):
    d2y_dx2 = dde.grad.hessian(y, x, i=0, j=0)
    return dde.grad.jacobian(d2y_dx2, x, i=0, j=0)
bc_obc2 = dde.icbc.OperatorBC(
    geom, boundary_third_derivative, boundary_right
)